### 1. 在abaqus中建立铰接模型，分析模态特性，确认铰接是否设置成功。
### 2. 导出质量矩阵和刚度矩阵

In [ ]:
#import packages
import time
import numpy as np
import capytaine as cpt
import scipy
from capytaine.io.mesh_writers import write_STL
import matplotlib.pyplot as plt
from scipy.linalg import block_diag
from scipy.linalg import eigh
import vtk
import logging
import xarray as xr
from capytaine.io.xarray import merge_complex_values
from capytaine.post_pro import rao
logging.basicConfig(level=logging.INFO, format='%(levelname)-8s: %(message)s')
# user defined functions
import DM_ShowNodes as DMshow
import DM_Reading as dm_r
import DM_Assemble as DM_A
import SEREP as SEREP

### 经典水弹性计算波浪问题

In [ ]:
# setting initial parameter, reading mass and stiffness matrix
num_nodes = 806 # 63 793
master_nodes = sorted([214, 208, 202, 196, 190, 617,611,605,599,593]) #[41, 39, 37, 35, 33, 31, 29, 27, 25, 23] DM_A.calculate_node_positions(424,6,10)
# master node of hinge model
# [214, 208, 202, 196, 190,617,611,605,599,593]
master_nodes_length = len(master_nodes)
number = 120
dataset = merge_complex_values(xr.open_dataset(f"E:\phd\Code\DM-FEM2D\HydrodynamicData\Yoga\DM10_{number}_direction0.nc"))
file_m = 'E:\phd\Code\DM-FEM2D\StructureData\Hinge\Job-1-2_MASS1.mtx'
file_k = 'E:\phd\Code\DM-FEM2D\StructureData\Hinge\Job-1-2_STIF1.mtx' 

omega = dataset.omega.values


In [ ]:
file_m = 'E:\phd\Code\DM-FEM2D\StructureData\Hinge\Job-1_MASS1_check.mtx'
file_k = 'E:\phd\Code\DM-FEM2D\StructureData\Hinge\Job-1_STIF1_check.mtx' 
# read mass and stiffness matrix
# M = dm_r.get_stiffness_matrix(file_m)
# k = dm_r.get_stiffness_matrix(file_k)
k = dm_r.get_stiffness_matrix(file_k)

In [ ]:
# reduce dofs
M_consistant= SEREP.reduce_dofs(M, num_nodes, [5])
k = SEREP.reduce_dofs(k, num_nodes, [5])

# transform mass matrix, beta=0 is consistant mass matrix
M = SEREP.transform_mass_matrix(M_consistant,beta=0)

# obtaine master dofs and slave dofs
MasterDofs, SlaveDofs = SEREP.separate_dofs(num_nodes, master_nodes)

# reduce matrix use SEREP
MR,KR,T = SEREP.SEREP(k, M, SlaveDofs, master_nodes)
# node displacement

In [ ]:
# node displacement
i = 0
# read hydrodynamic data
added_mass = dataset['added_mass'][i].values
radiation_damping = dataset['radiation_damping'][i].values
inertia_matrix = dataset['inertia_matrix'].values
# hydrostatic_stiffness = dataset['hydrostatic_stiffness'].values
F_w = dataset['Froude_Krylov_force'][i].values + dataset['diffraction_force'][i].values

# REDUCE THE MATRICES
added_mass = SEREP.reduce_dofs(added_mass,master_nodes_length,[5])
radiation_damping = SEREP.reduce_dofs(radiation_damping,master_nodes_length,[5])
# hyrostatic stiffness or fem spring stiffness
# 1. choice hydrostatic stiffness
hydrostatic_stiffness = dataset['hydrostatic_stiffness'].values
hydrostatic_stiffness = SEREP.reduce_dofs(hydrostatic_stiffness,master_nodes_length,[5])

inertia_matrix = SEREP.reduce_dofs(inertia_matrix,master_nodes_length,[5])
F_w = SEREP.reduce_force_matrix_dofs(F_w, master_nodes_length, 5).reshape(1,5*master_nodes_length)

# Generate the system matrices
mass = added_mass + MR
damping = radiation_damping
stiffness =  hydrostatic_stiffness + KR
# F_w = reverse_load_matrix(F_w, num_dofs=5)
# Solve in frequency domain
master_displacement = DM_A.solve_frequency_domain(mass, damping, stiffness, F_w, omega[i])
global_displacement_disorder = T @ master_displacement
global_displacement = SEREP.reorder_displacement_matrix(global_displacement_disorder, MasterDofs, SlaveDofs)

mid = global_displacement[2::5,:]


In [ ]:
displacement = np.array(mid)


In [ ]:
a1 = displacement[0:403]
a2 = displacement[403:806]

In [ ]:
A1 = a1.reshape(13,31)
A2 = a2.reshape(13,31)

In [ ]:
dis = np.hstack((A2,A1))
plt.imshow(abs(dis))
plt.colorbar()

In [ ]:
plt.plot(abs(dis[2,:]))

In [ ]:
plt.plot(abs(master_displacement[2::5,0]))

In [ ]:
for i in [214, 208, 202, 196, 190]:
    print(i+403)

In [ ]:
plt.imshow(abs(displacement))

In [ ]:
x = np.linspace(0, 1, 61)
plt.plot(x, abs(displacement[7,:]))
import pandas as pd
df = pd.read_csv(r'E:\phd\Code\DM-FEM2D\FEM_Reduce\Hinge\Default Dataset.csv')
plt.plot(df.iloc[:, 0],df.iloc[:, 1])
plt.ylim(0,1.6)
plt.legend(['RODM','FEM'])

In [ ]:
plt.imshow(abs(displacement))

In [ ]:
displacement = np.array(mid).reshape(13,61)
plt.imshow(abs(displacement))
plt.colorbar()

In [ ]:
def calculate_column_node_indices(column_number, nodes_per_row, rows_per_column):
    """
    计算给定列的节点编号。

    :param column_number: int，指定要计算的列号（从1开始计数）
    :param nodes_per_row: int，每行的节点数
    :param rows_per_column: int，每列的节点数
    :return: list，该列的所有节点编号
    """
    # 验证输入
    if column_number < 1 or column_number > nodes_per_row:
        raise ValueError("列号超出范围")
    
    # 计算该列的所有节点编号
    column_indices = [(row_index - 1) * nodes_per_row + column_number
                      for row_index in range(1, rows_per_column + 1)]
    return column_indices

# 使用示例
nodes_per_row = 61
rows_per_column = 13
column_number = 31  # 例如，计算第一列的节点编号
column_node_indices = calculate_column_node_indices(column_number, nodes_per_row, rows_per_column)
print("节点编号：", column_node_indices)


In [ ]:
def generate_elements(node_list_1, node_list_2):
    """
    根据提供的两个节点列表生成单元信息。

    参数:
    node_list_1 : list of int
        第一个节点列表。
    node_list_2 : list of int
        第二个节点列表。

    返回:
    list of list of int
        生成的单元信息。
    """
    elements = []
    # 我们将循环通过节点，除了最后一个，以便将每组与下一组配对
    for i in range(len(node_list_1) - 1):  # 减1是因为我们在向前看一个
        n1 = node_list_1[i]
        n2 = node_list_2[i]
        n3 = node_list_1[i + 1]
        n4 = node_list_2[i + 1]
        elements.append([n1, n2, n3, n4])
    
    return elements


def update_global_stiffness_matrix(k_total, k_element, elements):
    """
    更新全局刚度矩阵，减去指定单元的刚度矩阵。

    参数:
    k_total : numpy.ndarray
        全局刚度矩阵。
    k_element : numpy.ndarray
        单元刚度矩阵。
    elements : list of list of int
        单元节点信息，每个列表项包含一个单元的节点编号。

    返回:
    numpy.ndarray
        更新后的全局刚度矩阵。
    """
    # 对每个单元进行遍历
    for elem in elements:
        # 计算全局索引
        global_indices = []
        for node in elem:
            global_indices.extend(range((node - 1) * 6, (node - 1) * 6 + 6))

        # 将 k_element 的值逐一减去到 k_total 的对应位置
        for i, gi in enumerate(global_indices):
            for j, gj in enumerate(global_indices):
                k_total[gi, gj] -= k_element[i, j]

    return k_total

import numpy as np

def add_hinge_connections(big_matrix, nodes_k1, nodes_k2, k_hinge):
    """
    在全局刚度矩阵中添加铰接关系。

    参数:
    big_matrix : numpy.ndarray
        全局刚度矩阵。
    nodes_k1 : list of int
        第一组节点编号。
    nodes_k2 : list of int
        第二组节点编号。
    k_hinge : float
        铰接自由度的刚度值。

    返回:
    None，直接修改 big_matrix 矩阵。
    """
    # 创建 KC 矩阵
    KC = np.diag([k_hinge, k_hinge, k_hinge, k_hinge, 0, k_hinge])
    negative_KC = -KC

    for node1, node2 in zip(nodes_k1, nodes_k2):
        # 计算在大矩阵中的索引位置
        index1 = (node1 - 1) * 6  # K_1 节点自由度起始位置
        index2 = (node2 - 1) * 6  # K_2 节点自由度起始位置

        # 在节点自身设置 KC
        big_matrix[index1:index1+6, index1:index1+6] += KC
        big_matrix[index2:index2+6, index2:index2+6] += KC

        # 设置两节点间的相互作用 -KC
        big_matrix[index1:index1+6, index2:index2+6] += negative_KC
        big_matrix[index2:index2+6, index1:index1+6] += negative_KC
    return big_matrix


# 示例用法
node_list_1 = [30, 91, 152, 213, 274, 335, 396, 457, 518, 579, 640, 701, 762]
node_list_2 = [31, 92, 153, 214, 275, 336, 397, 458, 519, 580, 641, 702, 763]

# 生成单元信息
elements = generate_elements(node_list_1, node_list_2)
file_path_total = 'E:\phd\Code\DM-FEM2D\StructureData\Job-1_55_STIF1.mtx'
file_path_element = 'E:\\phd\\Code\\DM-FEM2D\\StructureData\\Hinge\\Element_stiff.mtx'
K_element = dm_r.read_element_stiffness_matrix(file_path_element)
K_total = dm_r.get_stiffness_matrix(file_path_total)
K_no_connect = update_global_stiffness_matrix(K_total,K_element,elements)
k_hinge = 10e15
K_hinge_connect = add_hinge_connections(K_total,node_list_1,node_list_2,k_hinge)

In [ ]:
file_path_total_mass = 'E:\phd\Code\DM-FEM2D\StructureData\Job-1_55_MASS1.mtx'
mass_total = dm_r.get_stiffness_matrix(file_path_total_mass)

In [ ]:
# reduce dofs
num_nodes = 793
mass_total= SEREP.reduce_dofs(mass_total, num_nodes, [5])
K_hinge_connect = SEREP.reduce_dofs(K_hinge_connect, num_nodes, [5])

In [ ]:
eigenvalues, eigenvectors = eigh(K_hinge_connect, mass_total)

In [ ]:
plt.imshow(eigenvectors[2::5,180].reshape(13,61))
plt.colorbar()

In [ ]:
plt.plot(eigenvectors[2::5,6].reshape(13,61)[7,:])